### Structured Output
Models can be requested to provide their response in a format matching a given schema. this is useful for ensuring the output can be easily parsed and used in subsequent procesing. Langchain supports multiple schema types and methods for enforcing structured output.

### Pydantic
Pydantic models provide the richest feature set with field validation, descriptions, and nested structures.

In [1]:
import os
from langchain.chat_models import init_chat_model
os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")
model = init_chat_model("groq:openai/gpt-oss-120b")
model

ChatGroq(metadata={'lc_versions': {'langchain-core': '1.6.12', 'langchain': '1.4.6'}}, profile={'name': 'GPT OSS 120B', 'release_date': '2025-08-05', 'last_updated': '2026-05-27', 'open_weights': True, 'max_input_tokens': 131072, 'max_output_tokens': 65536, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'structured_output': True, 'attachment': False, 'temperature': True}, client=<groq.resources.chat.completions.Completions object at 0x78a318d430e0>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x78a318d43e00>, model_name='openai/gpt-oss-120b', model_kwargs={}, groq_api_key=SecretStr('**********'))

In [ ]:
from pydantic import BaseModel,Field

class Movie(BaseModel):
    title:str=Field(description="The title of the movie")
    year:int=Field(description="This year the movie was released")
    director:str=Field(description="The director of the movie")
    rating:float=Field(description="The movies rating out of 10")

In [3]:
model_with_structure=model.with_structured_output(Movie)
model_with_structure

_ChatModelBinding(bound=ChatGroq(metadata={'lc_versions': {'langchain-core': '1.6.12', 'langchain': '1.4.6'}}, profile={'name': 'GPT OSS 120B', 'release_date': '2025-08-05', 'last_updated': '2026-05-27', 'open_weights': True, 'max_input_tokens': 131072, 'max_output_tokens': 65536, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'structured_output': True, 'attachment': False, 'temperature': True}, client=<groq.resources.chat.completions.Completions object at 0x78a318d430e0>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x78a318d43e00>, model_name='openai/gpt-oss-120b', model_kwargs={}, groq_api_key=SecretStr('**********')), kwargs={'tools': [{'type': 'function', 'function': {'name': 'Movie', 'description': '', 'parameters': {'properties': {'title': {'description': 'The title of 

In [4]:
model.invoke("Provide details about the movie Inception")

AIMessage(content='**Inception (2010) – Quick Reference**\n\n| Item | Details |\n|------|---------|\n| **Title** | *Inception* |\n| **Release Date** | July\u202f16\u202f,\u202f2010 (United States) |\n| **Running Time** | 148\u202fminutes |\n| **Genre** | Science‑fiction, Action, Thriller, Heist |\n| **Director / Writer** | Christopher Nolan |\n| **Producer(s)** | Emma Thomas, Christopher Nolan, Michael Caine (executive), Charles Roven (executive) |\n| **Production Companies** | Syncopy Inc., Warner Bros. Pictures, Legendary Pictures |\n| **Budget** | Approx. **$160\u202fmillion** |\n| **Box‑Office Gross** | $836\u202fmillion worldwide (≈\u202f$829\u202fmillion in theatrical receipts, $7\u202fmillion from re‑releases) |\n| **MPAA Rating** | PG‑13 (Violence, Some Language, Brief Sexual Content) |\n\n---\n\n### Main Cast\n\n| Actor / Actress | Character | Brief Note |\n|-----------------|-----------|------------|\n| **Leonardo DiCaprio** | Dom Cobb | A skilled “extractor” who steals secre

In [5]:
response= model_with_structure.invoke("Provide details about the movie Inception")
response

Movie(title='Inception', year=2010, director='Christopher Nolan', rating=8.8)

In [6]:
from pydantic import BaseModel,Field

class Movie(BaseModel):
    """A movie with details."""
    title:str=Field(..., description="The title of the movie")
    year:int=Field(..., description="This year the movie was released")
    director:str=Field(..., description="The director of the movie")
    rating:float=Field(..., description="The movie's rating out of 10")

model_with_structure = model.with_structured_output(Movie,include_raw=True)

response = model_with_structure.invoke("Provide details about the movie Inception")
response

{'raw': AIMessage(content='', additional_kwargs={'reasoning_content': 'User wants details about the movie Inception. We should call function Movie with appropriate fields. Need director: Christopher Nolan, rating (maybe 8.8), title: Inception, year: 2010. Provide details.', 'tool_calls': [{'id': 'fc_7227d998-578b-4d47-bac4-9dd1403dd395', 'function': {'arguments': '{"director":"Christopher Nolan","rating":8.8,"title":"Inception","year":2010}', 'name': 'Movie'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 99, 'prompt_tokens': 164, 'total_tokens': 263, 'completion_time': 0.208716329, 'completion_tokens_details': {'reasoning_tokens': 47}, 'prompt_time': 0.007364822, 'prompt_tokens_details': None, 'queue_time': 0.206312597, 'total_time': 0.216081151}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_803c0ba83d', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01a16e7e-aa45-7

### Nested Structure


In [8]:
from pydantic import BaseModel, Field

class Actor(BaseModel):
    name:str
    role:str

class MovieDetails (BaseModel):
    title:str
    year:int
    cast:list[Actor]
    genres: list[str]
    budget: float | None = Field(None, description="Budget in millions USD")

model_with_structure = model.with_structured_output(MovieDetails)
response = model_with_structure.invoke("Provide details about the movie Inception")
response

MovieDetails(title='Inception', year=2010, cast=[Actor(name='Leonardo DiCaprio', role='Dom Cobb'), Actor(name='Joseph Gordon-Levitt', role='Arthur'), Actor(name='Elliot Page', role='Ariadne'), Actor(name='Tom Hardy', role='Eames'), Actor(name='Ken Watanabe', role='Saito'), Actor(name='Cillian Murphy', role='Robert Fischer'), Actor(name='Marion Cotillard', role='Mal'), Actor(name='Michael Caine', role='Professor Stephen Miles')], genres=['Action', 'Adventure', 'Sci-Fi', 'Thriller'], budget=160000000.0)

### TypedDict
TypedDict provides a simpler alternative using Python's built-in typing, ideal when you don't need runtime validation


In [ ]:
from typing_extensions import TypedDict,Annotated

class MovieDict (TypedDict):
    """A movie with details."""
    title:Annotated[str,..., "The title of the movie"]
    year:Annotated[int,..., "This year the movie was released"]
    director:Annotated[str,..., "The director of the movie"]
    rating:Annotated[float,..., "The movie's rating out of 10"]

model_with_typeddict = model.with_structured_output(MovieDict)
response = model_with_typeddict.invoke("Please provide the details of the movie Avengers")
response


{'director': 'Joss Whedon', 'rating': 8, 'title': 'The Avengers', 'year': 2012}

In [ ]:
from typing_extensions import TypedDict

class Actor(TypedDict):
    name:str
    role:str

class MovieDetails (TypedDict):
    title:str
    year:int
    cast:list[Actor]
    genres: list[str]
    budget: float

model_with_typeddict = model.with_structured_output(MovieDetails)
response = model_with_typeddict.invoke("Provide details about the movie Avengers")
response

{'budget': 220000000,
 'cast': [{'name': 'Robert Downey Jr.', 'role': 'Tony Stark / Iron Man'},
  {'name': 'Chris Evans', 'role': 'Steve Rogers / Captain America'},
  {'name': 'Mark Ruffalo', 'role': 'Bruce Banner / Hulk'},
  {'name': 'Chris Hemsworth', 'role': 'Thor'},
  {'name': 'Scarlett Johansson', 'role': 'Natasha Romanoff / Black Widow'},
  {'name': 'Jeremy Renner', 'role': 'Clint Barton / Hawkeye'},
  {'name': 'Tom Hiddleston', 'role': 'Loki'},
  {'name': 'Samuel L. Jackson', 'role': 'Nick Fury'}],
 'genres': ['Action', 'Adventure', 'Sci-Fi', 'Superhero'],
 'title': 'The Avengers',
 'year': 2012}

In [20]:
model.profile

{'name': 'GPT OSS 120B',
 'release_date': '2025-08-05',
 'last_updated': '2026-05-27',
 'open_weights': True,
 'max_input_tokens': 131072,
 'max_output_tokens': 65536,
 'text_inputs': True,
 'image_inputs': False,
 'audio_inputs': False,
 'video_inputs': False,
 'text_outputs': True,
 'image_outputs': False,
 'audio_outputs': False,
 'video_outputs': False,
 'reasoning_output': True,
 'tool_calling': True,
 'structured_output': True,
 'attachment': False,
 'temperature': True}

### Data Classes
A data class is a class tipically containing mainly data, although there aren't really any restrictions. You create it using the @dataclass decorater 

In [22]:
from pydantic import BaseModel, Field
from langchain.agents import create_agent

class ContactInfo(BaseModel):
    """Contact information for a person."""
    name: str = Field(description="The name of the person")
    email:str = Field(description="The email address of the person")
    phone:str = Field(description="The phone number of the person")

agent = create_agent(
    model="gpt-5",
    response_format= ContactInfo #Auto-selects ProviderStrategy
)

result = agent.invoke({
    "messages":[{"role":"user","content":"Extract contact info from: Jhon Doe, jhon@example.com, (555) 123-4567"}]
})

result


{'messages': [HumanMessage(content='Extract contact info from: Jhon Doe, jhon@example.com, (555) 123-4567', additional_kwargs={}, response_metadata={}, id='cde733cf-b55c-40c9-bf11-b2774d31495a'),
  AIMessage(content='{"name":"Jhon Doe","email":"jhon@example.com","phone":"(555) 123-4567"}', additional_kwargs={'parsed': None, 'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 678, 'prompt_tokens': 206, 'total_tokens': 884, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 640, 'rejected_prediction_tokens': 0, 'text_tokens': None}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}}, 'model_provider': 'openai', 'model_name': 'gpt-5-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-EcadRyzlODX6TDrfzSrkk9P4n70H3', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--01a16f24-0bb8-7719-9b18-9c05486

In [23]:
print(result["structured_response"])

name='Jhon Doe' email='jhon@example.com' phone='(555) 123-4567'


In [ ]:
#Typedict
from typing_extensions import TypedDict
from langchain.agents import create_agent

class ContactInfo(TypedDict):
    """Contact information for a person."""
    name: str 
    email:str 
    phone:str 

agent = create_agent(
    model="gpt-5",
    response_format= ContactInfo #Auto-selects ProviderStrategy
)

result = agent.invoke({
    "messages":[{"role":"user","content":"Extract contact info from: Jhon Doe, jhon@example.com, (555) 123-4567"}]
})

print(result["structured_response"])

{'name': 'Jhon Doe', 'email': 'jhon@example.com', 'phone': '(555) 123-4567'}


In [26]:
## Dataclass 
from dataclasses import dataclass
from langchain.agents import create_agent

@dataclass
class ContactInfo:
    """Contact information for a person."""
    name: str 
    email:str 
    phone:str 

agent = create_agent(
    model="gpt-5",
    response_format= ContactInfo #Auto-selects ProviderStrategy
)

result = agent.invoke({
    "messages":[{"role":"user","content":"Extract contact info from: Jhon Doe, jhon@example.com, (555) 123-4567"}]
})

result["structured_response"]

ContactInfo(name='Jhon Doe', email='jhon@example.com', phone='(555) 123-4567')